[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees/cours/seance2_cours.ipynb)

# Séance 2.2 — Nettoyer des données réelles

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- repérer les sept défauts classiques d'un fichier réel
- convertir du texte en nombres et en dates
- traiter les valeurs manquantes en connaissance de cause
- supprimer les doublons et écarter les valeurs aberrantes
- construire un pipeline de nettoyage qu'on peut rejouer

## La séance précédente vous a menti

`ventes.csv` était **impeccable** : pas un trou, pas un doublon, des types
corrects. Ça n'arrive jamais.

Voici le même détaillant, mais l'export tel qu'il sort vraiment du système :
`ventes_sale.csv`.

> 🎯 **Votre mission de la séance :** transformer ce fichier en données
> exploitables, et savoir dire **combien de lignes** vous avez perdues au
> passage et **pourquoi**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

In [ ]:
sale = pd.read_csv(BASE + "ventes_sale.csv")   ## le fichier brut

print(sale.shape)   ## (lignes, colonnes)
sale.head(5)        ## cinq lignes suffisent a reperer l'essentiel

Prenez 30 secondes pour regarder ces cinq lignes. Qu'est-ce qui cloche ?

In [ ]:
sale.info()   ## regarder surtout la colonne Dtype

### Le diagnostic

`info()` révèle déjà deux problèmes graves :

- **`prix` est de type `object`** — c'est du **texte**, pas un nombre. On ne
  peut donc rien calculer avec. Coupable : `2,08 EUR`.
- **`date` est de type `object`** — du texte aussi. Impossible de demander
  « quel mois ? ».

Et un troisième, visible sur le `non-null` :

- **`client_id` a des trous.**

Il y en a quatre autres qu'`info()` ne montre pas. On va les débusquer.

## Défaut 1 — Les doublons

**On commence toujours par là.** Si vous filtrez d'abord, les doublons se
propagent dans tous vos comptages intermédiaires et vous ne saurez plus ce
que vous avez perdu.

In [ ]:
# duplicated() marque True chaque ligne deja vue plus haut
print("lignes strictement identiques :", sale.duplicated().sum())

251 lignes en double. Un client ne passe pas deux fois exactement la même
commande, à la même seconde, pour le même produit : c'est un **bug d'export**.

> ⚠️ Nuance importante : ici les lignes sont **strictement identiques sur
> toutes les colonnes**, donc on peut supprimer. Si seul le `cmd_id` était en
> double, ce pourrait être un vrai client commandant deux fois le même
> article. Vérifiez toujours **sur quelles colonnes** porte le doublon avant
> de supprimer.

In [ ]:
avant = len(sale)                        ## on note le point de depart
propre = sale.drop_duplicates().copy()   ## .copy() : un vrai tableau a soi

print(avant, "->", len(propre))          ## toujours mesurer ce qu'on retire

> 💡 Le `.copy()` dit à pandas : « fais-moi un vrai tableau indépendant ».
> Sans lui, il vous avertira plus tard que vous modifiez peut-être une simple
> vue du tableau d'origine. Prenez l'habitude de l'ajouter après un filtrage.

## Défaut 2 — Les valeurs manquantes

La commande à taper devant n'importe quel fichier :

In [ ]:
propre.isna().sum()   ## un compte de trous, colonne par colonne

407 lignes sans `client_id`. **Que faire ?**

Il n'y a pas de réponse universelle. Il y a une question à se poser :
**pourquoi cette valeur manque-t-elle ?**

Ici, probablement des ventes sans compte client (achat en magasin, commande
invitée). Donc :

| Votre question | La bonne décision |
|---|---|
| « Combien mes clients dépensent-ils ? » | **Supprimer** ces lignes : elles n'ont pas de client |
| « Quel est mon chiffre d'affaires total ? » | **Les garder** : ce sont de vraies ventes, les retirer fausserait le total |

> ⚠️ **Le piège à ne jamais commettre :** `fillna(0)` sur un identifiant. Vous
> créeriez un « client 0 » fantôme qui regrouperait 407 ventes sans aucun
> rapport entre elles. Remplir une valeur manquante, c'est **inventer une
> donnée** — ne le faites que si vous pouvez le justifier.

In [ ]:
# Notre question portera sur les clients : on supprime ces lignes,
# mais on note combien on en perd.
avant = len(propre)
propre = propre.dropna(subset=["client_id"]).copy()   ## cette colonne seule

print(avant, "->", len(propre), f"({avant - len(propre)} lignes retirees)")

## Défaut 3 — Des nombres stockés en texte

C'est le défaut le plus courant, et le plus sournois.

In [ ]:
propre["prix"].head(4)   ## du texte, pas des nombres

Deux problèmes dans une seule colonne :

1. Le suffixe **` EUR`** sur certaines valeurs.
2. La **virgule** comme séparateur décimal — convention française, alors que
   Python attend un point.

Trois étapes, dans cet ordre :

In [ ]:
# .str donne acces aux operations sur du texte, colonne entiere d'un coup
prix_txt = propre["prix"].str.replace(" EUR", "", regex=False)   ## 1. l'unite
prix_txt = prix_txt.str.replace(",", ".", regex=False)           ## 2. la virgule

propre["prix"] = pd.to_numeric(prix_txt, errors="coerce")   ## 3. la conversion
propre["prix"].head(4)   ## le type a change : ce sont des nombres

**`errors="coerce"`** veut dire : *« si tu n'arrives pas à convertir une
valeur, mets `NaN` au lieu de tout faire planter »*. C'est très pratique —
et très dangereux si on ne vérifie pas ensuite.

In [ ]:
# Reflexe obligatoire apres un coerce : combien de valeurs ont ete perdues ?
print("prix non convertis :", propre["prix"].isna().sum())

Zéro. Notre conversion est propre. **Faites systématiquement cette
vérification** : sans elle, vous pourriez transformer silencieusement 3 000
prix en `NaN` et ne vous en apercevoir qu'en présentant vos résultats.

## Défaut 4 — Les dates

Le plus piégeux. On procède en trois temps.

**Temps 1 :** la façon naïve.

In [ ]:
# Cellule volontairement fausse : lisez le message d'erreur
pd.to_datetime(propre["date"])   ## sans format, pandas devine... et echoue

`ValueError: time data "24-11-2011" doesn't match format "%d/%m/%Y"`

Le fichier mélange deux écritures : `14/11/2011` et `24-11-2011`. pandas veut
un format unique. Le message suggère lui-même la solution : `format="mixed"`.

**Temps 2 :** on ajoute `format="mixed"`.

In [ ]:
essai = pd.to_datetime(propre["date"], format="mixed")   ## deux ecritures

print("date la plus ancienne :", essai.min())
print("date la plus recente  :", essai.max())

Plus d'erreur. Mais regardez le résultat : **il est faux.**

Nos données couvrent décembre 2010 à décembre 2011. pandas annonce janvier
2010. Pourquoi ? Parce que `01/12/2010` a été lu **à l'américaine** : mois
d'abord, donc le 12 janvier. En français, c'est le 1er décembre.

**Temps 3 :** on impose la lecture française avec `dayfirst=True`.

In [ ]:
# dayfirst=True : lecture francaise, le jour avant le mois
propre["date"] = pd.to_datetime(propre["date"], format="mixed", dayfirst=True)

print("date la plus ancienne :", propre["date"].min())
print("date la plus recente  :", propre["date"].max())

> ⚠️ **Le point le plus important de la séance.** L'étape 1 produisait une
> **erreur bruyante** : gênante, mais elle vous arrête. L'étape 2 produisait
> une **erreur silencieuse** : le code tourne, les chiffres s'affichent, et
> ils sont faux. C'est de très loin la plus dangereuse.
>
> Après toute conversion, **vérifiez que le résultat est plausible** :
> `.min()`, `.max()`, un `head()`. Trente secondes qui vous éviteront de
> présenter des chiffres faux.

Une fois la colonne convertie en date, `.dt` ouvre tout :

In [ ]:
propre["mois"] = propre["date"].dt.month           ## .dt = boite a outils
propre["jour_sem"] = propre["date"].dt.dayofweek   ## 0 = lundi, 6 = dimanche

propre[["date", "mois", "jour_sem"]].head(3)

## Défaut 5 — Du texte incohérent

In [ ]:
print("nombre de categories distinctes :", propre["categorie"].nunique())
propre["categorie"].unique()[:8]

24 catégories, alors qu'il n'en existe que 8. Regardez bien : `' cuisine'`
avec un espace devant, `'CUISINE'` en majuscules, `'cuisine'`. Pour pandas,
ce sont **trois catégories différentes** — et tout `groupby` sur cette colonne
donnerait n'importe quoi.

> ⚠️ L'espace en début de chaîne est **invisible à l'écran**. C'est ce qui
> rend ce défaut particulièrement traître.

In [ ]:
# .str.strip() enleve les espaces au bord, .str.lower() met en minuscules
propre["categorie"] = propre["categorie"].str.strip().str.lower()   ## 24 -> 8

print("apres nettoyage :", propre["categorie"].nunique(), "categories")

## Défaut 6 — Les valeurs aberrantes

In [ ]:
propre["qte"].describe().round(1)   ## regarder min et max avant tout

Un minimum **négatif** et un maximum à **99 999**. Deux anomalies, mais elles
n'ont rien à voir :

- **`qte` négatif** : ce sont des **retours**. Ce n'est pas une erreur, c'est
  une information métier. On les écarte du calcul de chiffre d'affaires, mais
  on ne les jette pas — un taux de retour, ça s'analyse.
- **`qte = 99999`** : personne ne commande 99 999 articles. C'est une saisie
  erronée, ou un code sentinelle. On l'écarte.

> Traiter ces deux cas de la même façon serait une faute d'analyse.

In [ ]:
retours = propre.query("qte < 0")   ## un retour, pas une erreur
print("retours :", len(retours), "lignes")
print("quantites aberrantes :", len(propre.query("qte >= 10000")), "lignes")

avant = len(propre)
propre = propre.query("qte > 0 and qte < 10000").copy()   ## "and" dans query
print(avant, "->", len(propre))

## Défaut 7 — Créer des catégories utiles

Le nettoyage n'est pas que de la réparation : c'est aussi le moment où on
enrichit. Sans jamais écrire de boucle.

### Deux cas — `np.where`

In [ ]:
propre["ca"] = propre["qte"] * propre["prix"]   ## calculable enfin

# np.where(condition, valeur_si_vrai, valeur_si_faux)
propre["type"] = np.where(propre["ca"] > 50, "grosse", "petite")   ## deux cas
propre["type"].value_counts()

### Plusieurs cas — `np.select`

In [ ]:
conditions = [propre["ca"] > 200, propre["ca"] > 50]   ## ordre = priorite
etiquettes = ["tres grosse", "grosse"]

propre["taille"] = np.select(conditions, etiquettes, default="petite")
propre["taille"].value_counts()

⚠️ `np.select` prend la **première** condition vraie. L'ordre compte : si on
mettait `> 50` en premier, aucune ligne ne serait jamais « très grosse ».

### Des tranches — `pd.cut`

In [ ]:
propre["gamme"] = pd.cut(propre["prix"],
                         bins=[0, 1, 5, 20, 10000],   ## 5 bornes -> 4 tranches
                         labels=["entree", "eco", "milieu", "premium"])
propre["gamme"].value_counts()

## Le bilan — la seule cellule à ne jamais oublier

Un nettoyage se solde toujours par un compte rendu. Sans lui, personne ne peut
juger si votre analyse tient debout.

In [ ]:
print("lignes au depart :", len(sale))
print("lignes conservees :", len(propre))
print("taux de perte     :", round(100 * (1 - len(propre) / len(sale)), 1), "%")
print()
print("dont : 251 doublons, 407 sans client, 107 retours, 14 quantites aberrantes")

**14,5 % de pertes, et chacune est justifiée.** Voilà ce qu'on présente à un
responsable — pas « j'ai nettoyé les données ».

Si vous aviez perdu 40 % des lignes, il faudrait tout reprendre : ce ne serait
plus un nettoyage mais une erreur de méthode.

---

## Ce que vous savez faire maintenant

| Le problème | La commande |
|---|---|
| repérer les manquants | `df.isna().sum()` |
| supprimer les lignes incomplètes | `df.dropna(subset=["client_id"])` |
| remplacer les manquants | `df["prix"].fillna(0)` |
| compter les doublons | `df.duplicated().sum()` |
| supprimer les doublons | `df.drop_duplicates()` |
| texte → nombre | `pd.to_numeric(col, errors="coerce")` |
| texte → date | `pd.to_datetime(col, format="mixed", dayfirst=True)` |
| extraire le mois | `df["date"].dt.month` |
| nettoyer du texte | `col.str.strip().str.lower()` |
| enlever un morceau de texte | `col.str.replace(" EUR", "")` |
| classer selon une condition | `np.where(cond, "oui", "non")` |
| classer selon plusieurs | `np.select([c1, c2], ["a", "b"], default="c")` |
| découper en tranches | `pd.cut(col, bins=[0, 10, 50, 1000])` |

## La règle d'or

**Le nettoyage est un pipeline, pas une série de bricolages.** Écrivez-le dans
l'ordre, de haut en bas, en repartant toujours du fichier brut. Le jour où on
vous livre le fichier du mois suivant, vous relancez le notebook et c'est fini.

Et **notez toujours combien de lignes vous perdez à chaque étape**. Un
nettoyage qui fait disparaître 40 % des données n'est pas un nettoyage, c'est
une erreur.